# Tree of Thoughts — Exploring Multiple Reasoning Paths
## Branch, Evaluate, Prune — Deliberate Multi-Path Reasoning
⏱ ~15 min

**Tree of Thoughts (ToT)** is a prompting framework that abandons the single left-to-right chain of reasoning in favour of a deliberate *search* over a tree of partial solutions. Instead of committing to one thought at each step, the model fans out multiple candidate thoughts, evaluates each, and either prunes dead ends or expands the most promising paths — the way a chess player considers several moves before choosing.

By the end of this workshop you will understand *why* ToT outperforms Chain-of-Thought on hard planning and reasoning tasks, *how* LangGraph's Send API makes fan-out a first-class pattern, and *how* to build, tune, and extend a full ToT pipeline from scratch.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — CoT vs ToT vs GoT: what changed and why |
| 2 | **Search Strategies** — BFS, DFS, and beam search over thought trees |
| 3 | **Thought Generation** — prompting for diverse, high-quality branches |
| 4 | **Evaluation** — scoring thoughts with a judge LLM and heuristics |
| 5 | **Full ToT Pipeline** — LangGraph Send API end-to-end |
| 6 | **Beam Search Extension** — keep top-K branches across rounds |
| 7 | **Debugging & Tuning** — score distributions, diversity, cost |
| ★ | **Multi-Level Trees** (bonus) |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the repo's `requirements.txt`
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)

### Key References
> Yao, S., Yu, D., Zhao, J., et al. (2023). *Tree of Thoughts: Deliberate Problem Solving with Large Language Models.* NeurIPS 2023. https://arxiv.org/abs/2305.10601
>
> Wei, J., Wang, X., Schuurmans, D., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
>
> Besta, M., Blach, N., Kubicek, A., et al. (2024). *Graph of Thoughts: Solving Elaborate Problems with Large Language Models.* AAAI 2024. https://arxiv.org/abs/2308.09687
>
> LangGraph Send API docs: https://langchain-ai.github.io/langgraph/concepts/low_level/#send

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
            "typing-extensions",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid-looking: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Concepts: CoT, ToT, and GoT

---

### The Problem with a Single Chain

Chain-of-Thought prompting (Wei et al., 2022) vastly improved LLM reasoning by asking the model to show its work step by step. But it still produces a **single linear path** through reasoning space. One mis-step early in the chain — a wrong assumption, a flawed decomposition — cascades forward and poisons the final answer. There is no backtracking.

Tree of Thoughts (Yao et al., 2023) frames problem-solving as a **search problem**:
- The **state** is the partial solution so far (a sequence of thoughts)
- **Actions** are candidate next thoughts
- A **value function** (the evaluator) scores how promising each state is
- A **search algorithm** (BFS, DFS, beam) navigates the tree

---

### Three Approaches Compared

| Approach | Path structure | Backtracking | Evaluator | Best for |
|----------|---------------|--------------|-----------|----------|
| **Chain-of-Thought** | Single linear chain | None | None | Straightforward reasoning |
| **Tree of Thoughts** | Tree — fan out, prune, expand | Yes | Per-node score | Hard planning, math, puzzles |
| **Graph of Thoughts** | Arbitrary DAG — merge branches, loops | Yes | Per-node + edge | Tasks needing synthesis across paths |

**When to use ToT:**

| Use ToT | Stick with CoT |
|---------|----------------|
| Answer requires exploring trade-offs | Single-step factual Q&A |
| Multiple valid strategies exist | Translation, summarisation |
| Early mistakes are hard to recover from | Simple arithmetic |
| Search space is small enough to evaluate | Latency is the top constraint |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Thought** | A coherent language sequence that is one step toward solving the problem |
| **Thought generator** | The LLM call that proposes candidate next thoughts |
| **Evaluator / Judge** | LLM or heuristic that scores the promise of a partial solution |
| **Beam width** | Number of top candidates kept at each search level |
| **Branch** | One root-to-leaf path through the tree (a full candidate solution) |
| **Pruning** | Discarding low-scoring branches to focus compute on promising ones |
| **Fan-out** | Spawning N independent thought branches from the same starting state |

---

### Tree of Thoughts Architecture

```
PROBLEM (root)
     │
     ├─── Thought A ──── score=3  ✗ pruned
     │
     ├─── Thought B ──── score=8  ✓ expand
     │         │
     │         ├── B.1 ── score=6  ✗ pruned
     │         │
     │         └── B.2 ── score=9  ✓ best path
     │                      │
     │                      └── FINAL ANSWER
     │
     └─── Thought C ──── score=5  ✗ pruned


LangGraph Send API — how fan-out is implemented:

  generate_branches(state)
         │
         │  returns list[Send]
         │
         ├── Send("score_branch", {branch_id=0, ...})  ──┐
         ├── Send("score_branch", {branch_id=1, ...})  ──┤  parallel
         └── Send("score_branch", {branch_id=2, ...})  ──┘
                                                          │
                           Annotated[list, operator.add] ←┘
                           accumulates all branch results
                                                          │
                                                    select_best
                                                          │
                                                      synthesize
                                                          │
                                                        END
```

> **Source**: Architecture adapted from Yao et al. (2023) — Tree of Thoughts, NeurIPS 2023. LangGraph fan-out idiom from the LangGraph Send API documentation.

---

## Part 2 — Search Strategies: BFS, DFS, and Beam Search

---

The original ToT paper (Yao et al., 2023) implements two classical search algorithms over the thought tree. Each has a different cost-quality profile.

### Search Strategy Comparison

| Strategy | How it works | Memory | LLM calls | Best for |
|----------|--------------|--------|-----------|----------|
| **BFS (breadth-first)** | Expand all nodes level-by-level; keep top-b at each level | O(b^d) | b × d | Shallow trees, compare alternatives at same depth |
| **DFS (depth-first)** | Follow one path to the end, backtrack on failure | O(d) | Varies | Deep trees, early termination when answer found |
| **Beam search** | Like BFS but keeps only top-k at each step | O(k) | k × d | Balances quality and cost; most practical |
| **Greedy (ToT-0)** | Always take the single best thought (no branching) | O(1) | d | Baseline — equivalent to best-of-N generation |

Where **b** = branching factor (thoughts per node), **d** = depth (reasoning steps), **k** = beam width.

### Practical Default

For most production ToT applications: **beam search with k=2–5 and d=2–3** balances quality against LLM cost. Going wider (b > 5) or deeper (d > 4) produces diminishing returns while multiplying API calls exponentially.

In [ ]:
# ─── 2-A: Simulate search strategies without LLM calls ───────────────────────
# Illustrates BFS vs DFS vs beam search traversal order over a pre-built tree.
# In practice the nodes are LLM-generated; here we use a hardcoded tree to show
# traversal and selection logic at zero cost.

from dataclasses import dataclass, field


@dataclass
class ThoughtNode:
    id: str
    thought: str
    score: float
    children: list = field(default_factory=list)


# Build a sample tree manually
root = ThoughtNode("root", "Design a learning recommendation system", 0)
a = ThoughtNode("A", "Focus on collaborative filtering (user-user similarity)", 7.2)
b = ThoughtNode("B", "Focus on content-based filtering (skill graph)", 8.5)
c = ThoughtNode("C", "Focus on rule-based curriculum mapping", 4.1)
a1 = ThoughtNode("A.1", "Track peer learning histories", 6.8)
a2 = ThoughtNode("A.2", "Find similar engineers by role + tenure", 7.9)
b1 = ThoughtNode("B.1", "Build a prerequisite skill DAG", 9.1)
b2 = ThoughtNode("B.2", "Tag content by concept difficulty", 7.6)
c1 = ThoughtNode("C.1", "Follow a fixed learning track per role", 3.8)

root.children = [a, b, c]
a.children = [a1, a2]
b.children = [b1, b2]
c.children = [c1]


def bfs_search(root_node, beam_width=None):
    frontier = [root_node]
    visited = []
    while frontier:
        next_level = []
        for node in frontier:
            visited.append(node)
            next_level.extend(node.children)
        # beam_width prunes at each level — trades recall for cost; None = full BFS
        if beam_width:
            next_level = sorted(next_level, key=lambda n: n.score, reverse=True)[:beam_width]
        frontier = next_level
    return visited


def dfs_search(node, visited=None):
    if visited is None:
        visited = []
    visited.append(node)
    for child in sorted(node.children, key=lambda n: n.score, reverse=True):
        dfs_search(child, visited)
    return visited


def level_of(node, candidates):
    if node == root:
        return 0
    if node in [a, b, c]:
        return 1
    return 2


print("=== BFS (full breadth-first) ===")
for n in bfs_search(root):
    indent = "  " * level_of(n, [])
    print(f"{indent}[{n.id}] score={n.score:.1f}  {n.thought[:55]}")

print()
print("=== Beam Search (beam_width=2) ===")
for n in bfs_search(root, beam_width=2):
    indent = "  " * level_of(n, [])
    print(f"{indent}[{n.id}] score={n.score:.1f}  {n.thought[:55]}")

print()
print("=== DFS (best-first depth-first) ===")
for n in dfs_search(root):
    indent = "  " * level_of(n, [])
    print(f"{indent}[{n.id}] score={n.score:.1f}  {n.thought[:55]}")

best_leaf = max([a1, a2, b1, b2, c1], key=lambda n: n.score)
print(f"\nGlobal best leaf: [{best_leaf.id}] score={best_leaf.score}  '{best_leaf.thought}'")

---

## Part 3 — Thought Generation: Prompting for Diverse Branches

---

The quality of ToT depends entirely on the **diversity** of thoughts generated at each node. If all N branches say the same thing, the search adds no value over standard generation.

### Two Generation Strategies

| Strategy | How it works | Temperature | When to use |
|----------|--------------|-------------|-------------|
| **Sample** | Run the same prompt N times; high temp produces variation | 0.7–1.0 | General problems, creative tasks |
| **Propose** | Single prompt that asks for N distinct approaches in one call | 0.5–0.8 | When you want explicit differentiation |

The **propose approach** is cheaper (one LLM call instead of N) but requires the model to explicitly enumerate different strategies. The **sample approach** is simpler and works well with high enough temperature.

The existing `src/tools.py` uses the **sample approach** with `temperature=0.8` and instructs each branch to approach from a different angle via its `branch_id`. This is the simplest production-ready strategy.

In [ ]:
# ─── 3-A: Thought generator — sample approach ─────────────────────────────────
# Shows how temperature and per-branch instruction produce diverse thoughts.

import operator
from typing import Annotated

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict

N_BRANCHES = 3

# temperature=0.8 pushes branches apart — the whole point of ToT is diverse paths
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)   # diverse
# temperature=0.1 keeps the evaluator stable — same thought should get consistent scores
judge_llm     = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)   # consistent

BRANCH_PROMPT = """You are exploring solution path {branch_id} of {n_branches} for this problem.
Approach it from a DIFFERENT angle than typical solutions.

Problem: {problem}

Generate a creative, well-reasoned thought (3-5 sentences). Be specific and concrete."""

SCORE_PROMPT = """Rate this thought on a scale of 0-10 for solving the problem.
Criteria: relevance (0-4), creativity (0-3), feasibility (0-3).

Problem: {problem}
Thought: {thought}

Respond with ONLY a number 0-10."""

SYNTHESIS_PROMPT = """You selected the best thought from a tree search. Now expand it into a complete answer.

Problem: {problem}
Best Thought: {thought}
(This was scored highest of {n} candidate thoughts)

Write a thorough, actionable response."""

sample_problem = "Design a system to recommend personalized learning paths for software engineers."

print(f"Problem: {sample_problem}")
print(f"Generating {N_BRANCHES} branches (temperature=0.8)...\n")

thoughts = []
for i in range(N_BRANCHES):
    prompt = BRANCH_PROMPT.format(branch_id=i + 1, n_branches=N_BRANCHES, problem=sample_problem)
    response = generator_llm.invoke([HumanMessage(content=prompt)])
    thoughts.append(response.content.strip())
    print(f"--- Branch {i + 1} ---")
    print(thoughts[-1])
    print()

In [ ]:
# ─── 3-B: Score the generated thoughts ───────────────────────────────────────
# Judge LLM at low temperature gives consistent 0-10 scores.

scores = []
for i, thought in enumerate(thoughts):
    prompt = SCORE_PROMPT.format(problem=sample_problem, thought=thought)
    result = judge_llm.invoke([HumanMessage(content=prompt)])
    raw = result.content.strip()
    try:
        score = int("".join(filter(str.isdigit, raw))[:2])
        score = min(10, max(0, score))
    except (ValueError, IndexError):
        score = 5
    scores.append(score)

print("Branch score summary:\n")
print(f"{'Branch':<8} {'Score':>6}  {'Bar':20}  {'Thought preview'}")
print("-" * 78)
for i, (thought, score) in enumerate(zip(thoughts, scores)):
    bar = "█" * score + "░" * (10 - score)
    preview = thought[:40].replace("\n", " ") + "..."
    print(f"  {i + 1:<6} {score:>6}/10  {bar}  {preview}")

best_idx = max(range(len(scores)), key=lambda i: scores[i])
print(f"\nBest branch: {best_idx + 1} (score={scores[best_idx]}/10)")

---

## Part 4 — Evaluator Types: Judging Partial Solutions

---

The evaluator is the heart of ToT — it determines which paths are worth exploring. Yao et al. (2023) propose two styles:

### Evaluator Comparison

| Evaluator type | How it works | Pros | Cons |
|----------------|--------------|------|------|
| **Value (score)** | LLM rates each thought independently 0-10 | Simple, independent | LLM cost per thought |
| **Vote** | LLM sees all thoughts and picks the best one | Relative comparison | Needs all thoughts first |
| **Heuristic** | Python function (word count, keyword presence) | Zero LLM cost | Task-specific, brittle |
| **Verifier** | Runs code or checks constraints programmatically | Ground truth | Only works for verifiable tasks |

The `src/tools.py` in this example uses the **value** style: each branch is scored independently by a low-temperature judge LLM. This is the most general-purpose approach.

### Scoring Rubric Design

A well-designed rubric breaks the 0-10 range into dimensions with explicit sub-scores:
- **Relevance** (0-4): Does the thought actually address the problem?
- **Creativity** (0-3): Is it a non-obvious angle?
- **Feasibility** (0-3): Could this actually be built or done?

Total: 0-10. Adjust the sub-weights to match your task.

In [ ]:
# ─── 4-A: Value vs Vote vs Heuristic evaluators side-by-side ─────────────────

# --- Value evaluator (score each independently) ---
# Value: N separate calls — each branch scored independently; scores are comparable across runs
def value_evaluate(problem: str, thought_list: list[str]) -> list[int]:
    result_scores = []
    for thought in thought_list:
        prompt = SCORE_PROMPT.format(problem=problem, thought=thought)
        result = judge_llm.invoke([HumanMessage(content=prompt)])
        raw = result.content.strip()
        try:
            score = int("".join(filter(str.isdigit, raw))[:2])
            score = min(10, max(0, score))
        except (ValueError, IndexError):
            score = 5
        result_scores.append(score)
    return result_scores


# Vote: 1 call with all branches — forces relative ranking; better when scores cluster near the same value
# --- Vote evaluator (pick best by comparison) ---
VOTE_PROMPT = """You will see {n} candidate thoughts for solving a problem.
Vote for the SINGLE best thought. Reply with ONLY the number (1, 2, or 3...).

Problem: {problem}

{numbered_thoughts}

Best thought number:"""


def vote_evaluate(problem: str, thought_list: list[str]) -> int:
    numbered = "\n\n".join(f"{i+1}. {t[:200]}" for i, t in enumerate(thought_list))
    prompt = VOTE_PROMPT.format(n=len(thought_list), problem=problem, numbered_thoughts=numbered)
    result = judge_llm.invoke([HumanMessage(content=prompt)])
    raw = result.content.strip()
    try:
        winner = int("".join(filter(str.isdigit, raw))) - 1
        winner = max(0, min(len(thought_list) - 1, winner))
    except (ValueError, IndexError):
        winner = 0
    return winner


# Heuristic: zero LLM cost — use when quality is structural (length, keyword density)
# --- Heuristic evaluator (zero LLM cost) ---
def heuristic_evaluate(thought: str, keywords: list[str]) -> int:
    words = thought.split()
    sentences = [s for s in thought.replace("!", ".").replace("?", ".").split(".") if s.strip()]
    word_score     = min(4, len(words) // 10)
    keyword_hits   = sum(1 for k in keywords if k.lower() in thought.lower())
    keyword_score  = min(3, keyword_hits)
    sentence_score = min(3, len(sentences))
    return word_score + keyword_score + sentence_score


kw = ["learning", "skill", "engineer", "path", "recommend", "personali", "data", "model"]

print("=== Value Evaluator (independent 0-10 scores) ===")
value_scores = value_evaluate(sample_problem, thoughts)
for i, (s, t) in enumerate(zip(value_scores, thoughts)):
    print(f"  Branch {i+1}: {s}/10  —  {t[:60]}...")
print(f"  Winner: Branch {value_scores.index(max(value_scores)) + 1}")

print()
print("=== Vote Evaluator (relative comparison — 1 LLM call) ===")
vote_winner = vote_evaluate(sample_problem, thoughts)
print(f"  Winner: Branch {vote_winner + 1}")
print(f"  Thought: {thoughts[vote_winner][:100]}...")

print()
print("=== Heuristic Evaluator (zero LLM cost) ===")
h_scores = [heuristic_evaluate(t, kw) for t in thoughts]
for i, (s, t) in enumerate(zip(h_scores, thoughts)):
    print(f"  Branch {i+1}: {s}/10  —  {t[:60]}...")
print(f"  Winner: Branch {h_scores.index(max(h_scores)) + 1}")

---

## Part 5 — Full ToT Pipeline with LangGraph Send API

---

### How the Send API Enables Fan-out

LangGraph's `Send` primitive lets a single node schedule N independent invocations of any target node, each with its own private state. The results are merged back into the root state via an `Annotated` reducer.

```python
# The fan-out entry point returns a list of Send objects:
def generate_branches(state: ToTState) -> list[Send]:
    return [
        Send("score_branch", {"problem": state["problem"], "branch_id": i, ...})
        for i in range(N_BRANCHES)
    ]

# The reducer on the root state accumulates results:
class ToTState(TypedDict):
    branches: Annotated[list[dict], operator.add]  # list concat — no race conditions
```

Each `score_branch` invocation runs independently, returns `{"branches": [{...}]}`, and `operator.add` concatenates all lists so the parent state ends up with all N results.

### Graph Topology

```
START
  │
  └─ generate_branches (conditional entry point)
           │  returns list[Send]
           ├── score_branch (branch 0) ──┐
           ├── score_branch (branch 1) ──┤  all run independently
           └── score_branch (branch 2) ──┘
                                          │
                                    select_best
                                          │
                                      synthesize
                                          │
                                         END
```

In [ ]:
# ─── 5-A: State definitions ───────────────────────────────────────────────────
# Two separate TypedDicts: one for each branch's private state, one for the root.

import operator
from typing import Annotated

from typing_extensions import TypedDict


class BranchState(TypedDict):
    """Private state for each parallel thought branch.

    Lives only inside a single `score_branch` invocation.
    Does NOT flow back to the root state directly — only via the
    return dict {"branches": [...]}, which the reducer accumulates.
    """
    problem: str
    branch_id: int
    thought: str    # filled in by the node
    score: int      # filled in by the node


class ToTState(TypedDict):
    """Root state shared across the full graph.

    # Annotated + operator.add = merge reducer: parallel branch results get list-concatenated, not overwritten
    `branches` uses Annotated with operator.add so that each parallel
    branch's return value is list-concatenated rather than overwriting.
    """
    problem: str
    branches: Annotated[list[dict], operator.add]   # accumulates ALL branch results
    best_thought: str
    final_answer: str


print("BranchState fields:", list(BranchState.__annotations__.keys()))
print("ToTState fields:   ", list(ToTState.__annotations__.keys()))
print()
print("Key insight: Annotated[list[dict], operator.add]")
print("  branch_0 returns {'branches': [r0]}")
print("  branch_1 returns {'branches': [r1]}")
print("  LangGraph applies operator.add → root state.branches == [r0, r1]")
print("  No race conditions, no locks — declarative merge.")

In [ ]:
# ─── 5-B: Graph nodes ─────────────────────────────────────────────────────────

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.types import Send

N_BRANCHES = 3
llm       = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)   # diverse thought generation
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)   # consistent scoring


def generate_branches(state: ToTState) -> list[Send]:
    """Conditional entry point: fan out N branches using the Send API.

    Returning list[Send] from a conditional entry point is the LangGraph
    idiom for parallel fan-out. Each Send schedules one independent
    invocation of 'score_branch' with its own BranchState.
    """
    # list[Send] triggers parallel fan-out — LangGraph spawns one score_branch invocation per Send
    return [
        Send(
            "score_branch",
            {
                "problem": state["problem"],
                "branch_id": i,
                "thought": "",
                "score": 0,
            },
        )
        for i in range(N_BRANCHES)
    ]


def score_branch(state: BranchState) -> dict:
    """Generate a thought for this branch, then score it with the judge.

    Returns {"branches": [result_dict]} — the Annotated reducer on ToTState
    concatenates this with results from the other parallel branches.
    """
    branch_prompt = BRANCH_PROMPT.format(
        branch_id=state["branch_id"] + 1,
        n_branches=N_BRANCHES,
        problem=state["problem"],
    )
    thought_result = llm.invoke([HumanMessage(content=branch_prompt)])
    thought = thought_result.content.strip()

    score_prompt = SCORE_PROMPT.format(problem=state["problem"], thought=thought)
    score_result = judge_llm.invoke([HumanMessage(content=score_prompt)])
    try:
        score = int("".join(filter(str.isdigit, score_result.content.strip()))[:2])
        score = min(10, max(0, score))
    except (ValueError, IndexError):
        score = 5

    print(f"  Branch {state['branch_id'] + 1}: score={score}/10 | {thought[:70]}...")
    return {"branches": [{"branch_id": state["branch_id"], "thought": thought, "score": score}]}


def select_best(state: ToTState) -> dict:
    """Pick the highest-scoring branch. Pure Python — no LLM needed."""
    best = max(state["branches"], key=lambda b: b["score"])
    print(f"  Best branch: {best['branch_id'] + 1} (score={best['score']}/10)")
    return {"best_thought": best["thought"]}


def synthesize(state: ToTState) -> dict:
    """Expand the best thought into a full, actionable final answer."""
    prompt = SYNTHESIS_PROMPT.format(
        problem=state["problem"],
        thought=state["best_thought"],
        n=N_BRANCHES,
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": result.content}


print("Nodes defined: generate_branches, score_branch, select_best, synthesize")

In [ ]:
# ─── 5-C: Compile the graph ───────────────────────────────────────────────────

from langgraph.graph import END, StateGraph


def create_workflow():
    graph = StateGraph(ToTState)
    graph.add_node("score_branch", score_branch)
    graph.add_node("select_best", select_best)
    graph.add_node("synthesize", synthesize)
    # set_conditional_entry_point: the router runs first and returns list[Send] to drive fan-out
    # Conditional entry point returns list[Send] — this IS the fan-out
    graph.set_conditional_entry_point(
        generate_branches,
        {"score_branch": "score_branch"},   # maps Send target → registered node
    )
    graph.add_edge("score_branch", "select_best")
    graph.add_edge("select_best", "synthesize")
    graph.add_edge("synthesize", END)
    # compile() locks the graph topology — after this, add_node/add_edge calls are invalid
    return graph.compile()


app = create_workflow()
print("Graph compiled successfully.")
print()
print("Graph edges:")
print("  START → [conditional] → score_branch (×N in parallel)")
print("  score_branch → select_best")
print("  select_best  → synthesize")
print("  synthesize   → END")

In [ ]:
# ─── 5-D: Run the full pipeline on two sample problems ───────────────────────

SAMPLE_PROBLEMS = [
    "Design a system to recommend personalized learning paths for software engineers.",
    "How should a startup prioritize features when resources are limited?",
]

results = []

for problem in SAMPLE_PROBLEMS:
    print(f"\nPROBLEM: {problem}")
    print("-" * 70)
    result = app.invoke(
        {"problem": problem, "branches": [], "best_thought": "", "final_answer": ""}
    )
    results.append(result)

    print(f"\n{'Branch':<10} {'Score':>8}  {'Bar':20}  {'Thought preview'}")
    print("-" * 72)
    for b in sorted(result["branches"], key=lambda x: x["score"], reverse=True):
        winner = " ← best" if b["thought"] == result["best_thought"] else ""
        bar = "█" * b["score"] + "░" * (10 - b["score"])
        preview = b["thought"][:40].replace("\n", " ") + "..."
        print(f"  Branch {b['branch_id'] + 1:<4} {b['score']:>6}/10  {bar}  {preview}{winner}")

    print("\nFinal Answer (first 400 chars):")
    print(result["final_answer"][:400])
    print()

---

## Part 6 — Beam Search Extension: Keep Top-K Branches

---

The single-round ToT above fans out N branches, scores them, and selects one. This is equivalent to **beam search with depth=1 and beam_width=1**. Real beam search keeps the **top-k** branches at each level and fans each of them out to the next level.

```
Level 0 (root problem)
     │
     ├── Branch A (score=8) ──┐
     ├── Branch B (score=9) ──┤  keep top-2 (beam_width=2)
     └── Branch C (score=5) ──┘  prune C
           │               │
Level 1   A.1 A.2 A.3    B.1 B.2 B.3     score each
                                          keep global top-2
Level 2   ...                             final synthesis
```

The cell below implements **one round of beam selection** — keeping top-k from already-generated branches — as a minimal extension of the graph above.

In [ ]:
# ─── 6-A: Beam selection — keep top-K after scoring ──────────────────────────

BEAM_WIDTH = 2

BEAM_SYNTHESIS_PROMPT = """You selected the top {k} thoughts from a tree search. Synthesize them.

Problem: {problem}

{ranked_thoughts}

Synthesize the best ideas from all {k} thoughts into one thorough, actionable response."""


# Beam search keeps top-K thoughts instead of 1 — reduces the chance of a single fluky winner
class BeamToTState(TypedDict):
    problem: str
    branches: Annotated[list[dict], operator.add]
    top_thoughts: list[str]    # top-k instead of single best
    final_answer: str


def select_top_k(state: BeamToTState) -> dict:
    """Keep the top BEAM_WIDTH branches instead of just one."""
    ranked = sorted(state["branches"], key=lambda b: b["score"], reverse=True)
    top_k = ranked[:BEAM_WIDTH]
    print(f"  Top-{BEAM_WIDTH} branches:")
    for b in top_k:
        print(f"    Branch {b['branch_id'] + 1}: score={b['score']}/10")
    return {"top_thoughts": [b["thought"] for b in top_k]}


def beam_synthesize(state: BeamToTState) -> dict:
    """Synthesize the top-K thoughts into a single comprehensive answer."""
    ranked_thoughts = "\n\n".join(
        f"Thought {i+1} (rank #{i+1}): {t[:300]}"
        for i, t in enumerate(state["top_thoughts"])
    )
    prompt = BEAM_SYNTHESIS_PROMPT.format(
        k=BEAM_WIDTH,
        problem=state["problem"],
        ranked_thoughts=ranked_thoughts,
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_answer": result.content}


def _generate_for_beam(state: BeamToTState) -> list[Send]:
    return [
        Send("score_branch", {"problem": state["problem"], "branch_id": i, "thought": "", "score": 0})
        for i in range(N_BRANCHES)
    ]


beam_graph = StateGraph(BeamToTState)
beam_graph.add_node("score_branch", score_branch)
beam_graph.add_node("select_top_k", select_top_k)
beam_graph.add_node("beam_synthesize", beam_synthesize)
beam_graph.set_conditional_entry_point(_generate_for_beam, {"score_branch": "score_branch"})
beam_graph.add_edge("score_branch", "select_top_k")
beam_graph.add_edge("select_top_k", "beam_synthesize")
beam_graph.add_edge("beam_synthesize", END)
# Same topology as ToTState graph — only the select and synthesize nodes differ
beam_app = beam_graph.compile()

print(f"Beam ToT compiled (beam_width={BEAM_WIDTH}, N_BRANCHES={N_BRANCHES}).")

beam_problem = "How should a startup prioritize features when resources are limited?"
print(f"\nPROBLEM: {beam_problem}")
print("-" * 70)
beam_result = beam_app.invoke(
    {"problem": beam_problem, "branches": [], "top_thoughts": [], "final_answer": ""}
)
print("\nBeam Synthesis Answer (first 500 chars):")
print(beam_result["final_answer"][:500])

---

## Part 7 — Debugging and Tuning

---

### Common ToT Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|------------|-----|
| **Branch collapse** | All branches say the same thing | Temperature too low | Raise temperature to 0.9+, add explicit diversity instruction |
| **Score bunching** | All scores cluster around 7/10 | Rubric too coarse | Rewrite rubric with explicit anchors per score level |
| **Synthesis ignores best** | Final answer doesn't reflect winning thought | Synthesis prompt too open | Explicitly quote the best thought in the synthesis prompt |
| **Judge inflation** | Every thought scores 9-10 | Judge lacks grounding | Add concrete negative examples to the scoring prompt |
| **Cost explosion** | Too many LLM calls | N_BRANCHES or depth too large | Use heuristic evaluator for intermediate levels |

### Diagnostics Checklist

1. Print all branch thoughts before scoring — check for semantic diversity
2. Print the raw judge output — verify it returns only a number
3. Check score distribution — median ~6-7, range spanning at least 3 points
4. Verify synthesis output references ideas from the best branch

In [ ]:
# ─── 7-A: Score distribution analysis ────────────────────────────────────────

import re
import statistics

diagnostic_problem = "Design a system to recommend personalized learning paths for software engineers."

print(f"Problem: {diagnostic_problem}")
print(f"Running {N_BRANCHES} branches for score distribution...\n")

diag_result = app.invoke(
    {"problem": diagnostic_problem, "branches": [], "best_thought": "", "final_answer": ""}
)

run_scores = [b["score"] for b in diag_result["branches"]]

print("Score distribution:")
print(f"  Values : {sorted(run_scores, reverse=True)}")
print(f"  Min    : {min(run_scores)}")
print(f"  Max    : {max(run_scores)}")
print(f"  Range  : {max(run_scores) - min(run_scores)}")
print(f"  Mean   : {statistics.mean(run_scores):.1f}")
if len(run_scores) > 1:
    print(f"  Stdev  : {statistics.stdev(run_scores):.2f}")
print()

spread = max(run_scores) - min(run_scores)
if spread < 2:
    print("WARNING: Score range < 2 — branches may be too similar.")
    print("Try: raise temperature, add stronger differentiation instruction.")
else:
    print(f"Score spread = {spread} — good differentiation between branches.")

In [ ]:
# ─── 7-B: Diversity check — Jaccard similarity between branch thoughts ────────
# Jaccard similarity: |A ∩ B| / |A ∪ B|  — lower = more diverse.
# No embedding cost — purely lexical.


def word_set(text: str) -> set:
    stopwords = {"the", "a", "an", "is", "it", "to", "of", "and", "or", "in", "for", "by", "with"}
    words = re.findall(r"[a-z]+", text.lower())
    return {w for w in words if w not in stopwords and len(w) > 2}


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


branch_thoughts = [b["thought"] for b in diag_result["branches"]]
branch_words    = [word_set(t) for t in branch_thoughts]

print("Pairwise Jaccard similarity between branch thoughts")
print("(lower = more diverse, 0.0 = no word overlap, 1.0 = identical)\n")

n = len(branch_thoughts)
similarities = []
for i in range(n):
    for j in range(i + 1, n):
        sim = jaccard(branch_words[i], branch_words[j])
        similarities.append(sim)
        flag = "⚠ similar" if sim > 0.4 else "✓ diverse"
        print(f"  Branch {i+1} vs Branch {j+1}: {sim:.3f}  {flag}")

print(f"\nMean similarity: {statistics.mean(similarities):.3f}")
if statistics.mean(similarities) > 0.4:
    print("Branches are semantically similar — consider increasing temperature.")
else:
    print("Good diversity across branches.")

---

## Exercises

---

### Exercise 1 — Scale N_BRANCHES and Measure the Trade-off

**Goal:** Understand the cost-quality trade-off of wider trees.

Run the pipeline with `N_BRANCHES = 2`, `4`, and `6` on the same problem. For each run record:
- The winning branch score
- The score range (max − min)
- The number of LLM calls (`N_BRANCHES × 2 + 1`)

**Hypothesis:** Higher N_BRANCHES improves the winning score up to a point, then plateaus. Where is the plateau?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
exercise_problem = "How do you design a fault-tolerant distributed caching layer?"

# TODO: run the pipeline with n_branches in [2, 4, 6]
# Use make_tot_app(n_branches) from the answer key below,
# or redefine generate_branches to capture n_branches.

# results_by_n = {}
# for n in [2, 4, 6]:
#     trial_app = make_tot_app(n)
#     r = trial_app.invoke({"problem": exercise_problem, "branches": [], "best_thought": "", "final_answer": ""})
#     best = max(r["branches"], key=lambda b: b["score"])
#     results_by_n[n] = {"best_score": best["score"], "range": max(b["score"] for b in r["branches"]) - min(b["score"] for b in r["branches"]), "llm_calls": n * 2 + 1}

print("Exercise 1: run make_tot_app(n) for n in [2, 4, 6], record best_score + range.")
print("LLM calls = N × 2 + 1 (generation + scoring + synthesis)")
for n in [2, 4, 6]:
    print(f"  n={n}: {n * 2 + 1} LLM calls")

### Exercise 2 — Replace the Judge with a Python Heuristic

**Goal:** Understand when an LLM evaluator is necessary vs when Python suffices.

Implement `heuristic_score_branch` below to score thoughts by:
- Word count (more specific thoughts tend to be longer)
- Presence of domain keywords (you define the list)
- Number of sentences (more reasoning steps = more thorough)

Then modify `score_branch` to call your heuristic instead of `judge_llm` and run the pipeline. Compare the heuristic winners against the LLM judge winners on the same problem.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

def heuristic_score_branch(thought: str, keywords: list[str]) -> int:
    """Score a thought using Python heuristics only (zero LLM cost).

    Returns an integer 0-10.
    """
    # TODO: implement a multi-factor score
    # Suggested factors:
    #   word_count_score  = min(4, len(thought.split()) // 10)
    #   keyword_score     = min(3, sum(1 for k in keywords if k.lower() in thought.lower()))
    #   sentence_score    = min(3, thought.count('.') + thought.count('!'))
    #   return word_count_score + keyword_score + sentence_score
    pass


domain_keywords = ["TODO: add keywords relevant to your problem"]

# TODO: create a modified score_branch that calls heuristic_score_branch
#       instead of judge_llm, rebuild the workflow, run, and compare winners.

print("Exercise 2: implement heuristic_score_branch, swap it for judge_llm, compare.")

### Exercise 3 — Two-Level Tree (Depth 2)

**Goal:** Extend the flat single-round ToT to a genuine two-level tree.

After `select_best` (which picks the top branch after level 1), add a second fan-out step:
1. Take the best thought from level 1 as a partial solution
2. Fan out 3 sub-branches that each *extend* that thought in a different direction
3. Score the sub-branches
4. Select the best sub-branch and synthesize

**Hint:** You need a second `Annotated[list, operator.add]` field in the state and a second conditional fan-out node that reads `state["best_thought"]`.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

EXTEND_PROMPT = """You are extending the following partial solution:

Partial solution: {partial}

Problem: {problem}

Extend this partial solution in direction {direction_id} of {n_directions}.
Focus on a DIFFERENT aspect than the other directions would.
Write 3-5 concrete sentences."""


class ExtendedToTState(TypedDict):
    problem: str
    branches: Annotated[list[dict], operator.add]          # level-1 branches
    best_thought: str                                       # winner from level 1
    sub_branches: Annotated[list[dict], operator.add]      # TODO: level-2 branches
    final_answer: str


# TODO: implement generate_sub_branches(state) -> list[Send]
# TODO: implement score_sub_branch(state) -> dict
# TODO: implement select_best_sub(state) -> dict
# TODO: wire the two-level graph

print("Exercise 3: implement a second fan-out layer from the best level-1 thought.")
print("Expected graph:")
print("  START → generate_branches → score_branch (×N) → select_best")
print("       → generate_sub_branches → score_sub_branch (×N) → select_best_sub → synthesize → END")
print(f"Total LLM calls (N={N_BRANCHES}): {N_BRANCHES * 4 + 1}")

---

## Part 8 ★ — Vote Evaluation and When to Use It (Bonus)

---

Vote evaluation forces the judge to **compare** all candidates before deciding, rather than rating each in isolation. This improves calibration when thoughts are close in quality — the independent 0-10 scores tend to bunch, but the judge can still pick a clear winner comparatively.

### When to use vote vs value

| Signal | Use value (0-10) | Use vote |
|--------|-----------------|----------|
| Thoughts are very different | ✓ independent scoring works | Overkill |
| Thoughts are similar (collapsed) | Scores bunch around 7 | ✓ forces comparison |
| N is large (N > 5) | ✓ cheaper | Expensive prompt |
| N is small (N ≤ 4) | Works fine | ✓ better calibration |

In [ ]:
# ─── 8-A: Vote evaluation on the diagnostic run ───────────────────────────────
# Reuses branches already generated in Part 7 — no extra generation cost.

VOTE_PROMPT_FULL = """You will compare {n} candidate thoughts for solving a problem.
Select the single BEST thought. Reply with ONLY the number (1 through {n}).

Problem: {problem}

{numbered_thoughts}

Best thought number:"""


def vote_best(problem: str, branches: list[dict]) -> dict:
    """Ask the judge to vote for the best branch in a single comparative LLM call."""
    numbered = "\n\n".join(
        f"{i+1}. {b['thought'][:300]}"
        for i, b in enumerate(branches)
    )
    prompt = VOTE_PROMPT_FULL.format(
        n=len(branches),
        problem=problem,
        numbered_thoughts=numbered,
    )
    result = judge_llm.invoke([HumanMessage(content=prompt)])
    raw = result.content.strip()
    try:
        winner_idx = int("".join(filter(str.isdigit, raw))) - 1
        winner_idx = max(0, min(len(branches) - 1, winner_idx))
    except (ValueError, IndexError):
        winner_idx = 0
    return branches[winner_idx]


print(f"Problem: {diagnostic_problem}")

print("\n=== Value Evaluator (independent 0-10 scores) ===")
value_winner = max(diag_result["branches"], key=lambda b: b["score"])
for b in sorted(diag_result["branches"], key=lambda b: b["score"], reverse=True):
    marker = " ← winner" if b is value_winner else ""
    print(f"  Branch {b['branch_id']+1}: {b['score']}/10{marker}")
print(f"  Winner thought: {value_winner['thought'][:80]}...")

print("\n=== Vote Evaluator (one comparative call) ===")
vote_winner = vote_best(diagnostic_problem, diag_result["branches"])
print(f"  Winner: Branch {vote_winner['branch_id']+1}")
print(f"  Winner thought: {vote_winner['thought'][:80]}...")

agree = value_winner["branch_id"] == vote_winner["branch_id"]
print(f"\nValue and vote agree: {agree}")
if not agree:
    print("Disagreement detected — the judges differ. Vote is often more reliable when thoughts are close.")

---

## What's Next?

You now have a working Tree of Thoughts pipeline, understand the search strategies, and have diagnostic tools for debugging branch collapse and score distribution. Here is where to go from here.

### Apply the fan-out pattern to retrieval
- **Example 26 — RAG Fusion** ([`../26-rag-fusion/`](../26-rag-fusion/)): the same `list[Send]` fan-out applied to parallel retrieval queries, merged with Reciprocal Rank Fusion. A natural next step if you want Send API in a RAG context.

### Run full multi-agent graphs
- **Example 6 — Multi-Agent Graph** ([`../6-multi-agent-graph/`](../6-multi-agent-graph/)): multiple specialist agents running in parallel subgraphs. The architectural cousin of multi-branch ToT.

### Add upfront planning
- **Example 37 — ReWOO Agent** ([`../37-rewoo-agent/`](../37-rewoo-agent/)): plan all tool calls before executing any — a complementary strategy to ToT's search-based approach.

### Push the tree deeper
- Complete Exercise 3 (depth-2 tree) and compare against the single-round output on a hard planning problem
- Try the original ToT benchmark tasks from Yao et al.: Game of 24, Creative Writing, Mini Crossword

### Production considerations
- **Cost control**: use `gpt-4o-mini` for generation and scoring, reserve `gpt-4o` for synthesis only
- **Caching**: cache branch thoughts keyed by `(problem, branch_id, seed)` for repeated queries
- **Async**: all LangGraph invocations support `ainvoke()` for non-blocking web server use
- **LangSmith tracing**: add `LANGCHAIN_API_KEY` to `.env` to trace every branch call in a visual timeline

---

*Workshop complete. The natural next example is 26 (RAG Fusion) — it reuses the Send API fan-out pattern in a retrieval context.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Scale N_BRANCHES

**Sample answer:**

LLM calls per run: `N_BRANCHES × 2 + 1` (N generation + N scoring + 1 synthesis).

Typical empirical finding:
- N=2: winning score often 7-8; limited diversity
- N=3: winning score 8-9; good balance of diversity and cost
- N=5: marginal improvement over N=3; cost increases 66%
- N=8+: diminishing returns; extra branches mostly duplicate earlier ideas

**Rule of thumb:** N=3–5 is the sweet spot. Beyond 5, you are paying for diversity the generator cannot reliably produce at a single temperature setting.

In [ ]:
# Sample solution for Exercise 1

def make_tot_app(n_branches: int):
    """Create a ToT pipeline with a configurable number of branches."""
    def _generate(state: ToTState) -> list[Send]:
        return [
            Send(
                "score_branch",
                {"problem": state["problem"], "branch_id": i, "thought": "", "score": 0},
            )
            for i in range(n_branches)
        ]

    g = StateGraph(ToTState)
    g.add_node("score_branch", score_branch)
    g.add_node("select_best", select_best)
    g.add_node("synthesize", synthesize)
    g.set_conditional_entry_point(_generate, {"score_branch": "score_branch"})
    g.add_edge("score_branch", "select_best")
    g.add_edge("select_best", "synthesize")
    g.add_edge("synthesize", END)
    return g.compile()


print("make_tot_app(n_branches) creates a fresh ToT pipeline with configurable branching.")
print("LLM calls per run:")
for n in [2, 3, 4, 6, 8]:
    print(f"  n={n}: {n * 2 + 1} calls")

### Exercise 2 — Heuristic Evaluator

**Sample answer:**

Heuristics work well when:
- Quality is structural (length, keyword density, format compliance)
- The problem domain has a clear vocabulary you can enumerate in advance
- You need to score hundreds of thoughts and LLM cost is prohibitive

Heuristics fail when:
- Quality is semantic (a short, precise thought may be better than a long rambling one)
- Keywords appear in bad thoughts as well as good ones
- The evaluator needs to understand implications, not just surface features

**Key finding:** heuristic and LLM judges agree roughly 60-70% of the time on typical problems. They diverge most when thought lengths are similar but semantic quality differs sharply.

In [ ]:
# Sample heuristic evaluator for Exercise 2

def heuristic_score_branch_solved(thought: str, keywords: list[str]) -> int:
    """Score a thought using Python heuristics — zero LLM cost."""
    words = thought.split()
    sentences = [s for s in thought.replace("!", ".").replace("?", ".").split(".") if s.strip()]
    word_score     = min(4, len(words) // 10)                                          # 0-4
    keyword_hits   = sum(1 for k in keywords if k.lower() in thought.lower())
    keyword_score  = min(3, keyword_hits)                                              # 0-3
    sentence_score = min(3, len(sentences))                                            # 0-3
    return word_score + keyword_score + sentence_score


sample_kw = ["learning", "skill", "engineer", "path", "recommend", "personali", "data", "model"]

print("Heuristic vs LLM judge on diagnostic branches:\n")
for b in diag_result["branches"]:
    h = heuristic_score_branch_solved(b["thought"], sample_kw)
    l = b["score"]
    diff = abs(h - l)
    agree = "✓" if diff <= 2 else "✗"
    print(f"  Branch {b['branch_id']+1}: heuristic={h}/10  llm={l}/10  {agree} diff={diff}")

h_best = max(diag_result["branches"], key=lambda b: heuristic_score_branch_solved(b["thought"], sample_kw))
l_best = max(diag_result["branches"], key=lambda b: b["score"])
print(f"\nHeuristic winner: Branch {h_best['branch_id']+1}")
print(f"LLM judge winner: Branch {l_best['branch_id']+1}")
print(f"Agree: {h_best['branch_id'] == l_best['branch_id']}")

### Exercise 3 — Two-Level Tree

**Sample answer:**

A depth-2 tree requires:
1. A second `Annotated[list, operator.add]` field in the state for sub-branches
2. A second conditional fan-out node (`generate_sub_branches`) reading `state["best_thought"]`
3. A second `score_sub_branch` node that extends the partial solution
4. Edges wiring level-1 tail → level-2 fan-out → level-2 scoring → select → synthesize

**LLM calls for depth-2:** `N × 2 (level 1) + N × 2 (level 2) + 1 (synthesis) = 4N + 1`

With N=3: 13 calls total vs 7 for depth-1. The output is typically more specific and actionable.

In [ ]:
# Sample skeleton for Exercise 3 — two-level tree

class SubBranchState(TypedDict):
    problem: str
    partial: str      # the level-1 best thought being extended
    branch_id: int
    thought: str
    score: int


class Depth2ToTState(TypedDict):
    problem: str
    branches: Annotated[list[dict], operator.add]        # level-1 branches
    best_thought: str                                     # winner from level 1
    sub_branches: Annotated[list[dict], operator.add]    # level-2 branches
    best_sub_thought: str                                # winner from level 2
    final_answer: str


def generate_sub_branches(state: Depth2ToTState) -> list[Send]:
    """Fan out N sub-branches from the best level-1 thought."""
    return [
        Send(
            "score_sub_branch",
            {
                "problem": state["problem"],
                "partial": state["best_thought"],
                "branch_id": i,
                "thought": "",
                "score": 0,
            },
        )
        for i in range(N_BRANCHES)
    ]


def score_sub_branch(state: SubBranchState) -> dict:
    """Extend the partial solution and score the extension."""
    prompt = EXTEND_PROMPT.format(
        partial=state["partial"],
        problem=state["problem"],
        direction_id=state["branch_id"] + 1,
        n_directions=N_BRANCHES,
    )
    thought_result = llm.invoke([HumanMessage(content=prompt)])
    thought = thought_result.content.strip()

    score_prompt = SCORE_PROMPT.format(problem=state["problem"], thought=thought)
    score_result = judge_llm.invoke([HumanMessage(content=score_prompt)])
    try:
        score = int("".join(filter(str.isdigit, score_result.content.strip()))[:2])
        score = min(10, max(0, score))
    except (ValueError, IndexError):
        score = 5

    print(f"  Sub-branch {state['branch_id'] + 1}: score={score}/10")
    return {"sub_branches": [{"branch_id": state["branch_id"], "thought": thought, "score": score}]}


def select_best_sub(state: Depth2ToTState) -> dict:
    best = max(state["sub_branches"], key=lambda b: b["score"])
    print(f"  Best sub-branch: {best['branch_id'] + 1} (score={best['score']}/10)")
    return {"best_sub_thought": best["thought"]}


print("Depth-2 skeleton defined. Wire with StateGraph to complete Exercise 3.")
print(f"Total LLM calls for depth-2 (N={N_BRANCHES}): {N_BRANCHES * 4 + 1}")

---

## Academic References

1. **Yao, S., Yu, D., Zhao, J., Shafran, I., Griffiths, T. L., Cao, Y., & Narasimhan, K. (2023).** *Tree of Thoughts: Deliberate Problem Solving with Large Language Models.* Advances in Neural Information Processing Systems (NeurIPS 2023). https://arxiv.org/abs/2305.10601

2. **Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q., & Zhou, D. (2022).** *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* Advances in Neural Information Processing Systems (NeurIPS 2022). https://arxiv.org/abs/2201.11903

3. **Besta, M., Blach, N., Kubicek, A., Gerstenberger, R., Gianinazzi, L., Gajda, J., Lehmann, T., Podstawski, M., Niewiadomski, H., Nyczyk, P., & Hoefler, T. (2024).** *Graph of Thoughts: Solving Elaborate Problems with Large Language Models.* AAAI 2024. https://arxiv.org/abs/2308.09687

4. **Liu, N. F., Lin, K., Hewitt, J., Paranjape, A., Bevilacqua, M., Petroni, F., & Liang, P. (2023).** *Lost in the Middle: How Language Models Use Long Contexts.* Transactions of the Association for Computational Linguistics. https://arxiv.org/abs/2307.03172

5. **LangGraph Send API documentation** — Fan-out with parallel node invocations: https://langchain-ai.github.io/langgraph/concepts/low_level/#send

6. **LangGraph Reducers documentation** — `Annotated` state merging: https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers